In [ ]:

# import pandas as pd
# import numpy as np
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler
# from sklearn.feature_selection import VarianceThreshold

# # ---------- 0) Load ----------
# path = "df_after_drop50.csv"   # 통합 데이터 경로
# df = pd.read_csv(path, low_memory=False)

# target_col = "기부여부"

# # ---------- 1) 결측치 50% 이상 컬럼 제거 ----------
# df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# # ---------- 2) 결측치 중앙값 대체 ----------
# imputer = SimpleImputer(strategy="median")
# df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# # ---------- 3) 타겟 분리 ----------
# df[target_col] = df[target_col].astype(int)
# X = df.drop(columns=[target_col])
# y = df[target_col]

# # ---------- 4) 저카디널리티(고유값 ≤ 10) 컬럼만 원핫 인코딩 ----------
# low_cardinality_cols = X.columns[X.nunique() <= 10]
# X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

# # ---------- 5) 분산이 0.01 이하인 컬럼 제거 ----------
# X = pd.DataFrame(VarianceThreshold(threshold=0.01).fit_transform(X))

# # ---------- 6) 다중공선성 제거 (상관계수 > 0.9) ----------
# corr_matrix = X.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# drop_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
# X.drop(columns=drop_cols, inplace=True)

# # ---------- 7) 스케일링 (StandardScaler 기준, 하나의 버전만 유지) ----------
# scaler = StandardScaler()
# X = pd.DataFrame(scaler.fit_transform(X))

# print(" 최종 데이터 Shape:", X.shape, "Target Shape:", y.shape)


In [ ]:
import pandas as pd

# 데이터 불러오기
path = "df_after_drop50.csv"  
df = pd.read_csv(path, low_memory=False)

# 1) 범주형 컬럼 찾기 (숫자가 아닌 dtype)
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("범주형 컬럼 개수:", len(cat_cols))
print("범주형 컬럼 목록:\n", cat_cols)

# 2) 각 범주형 컬럼별 고유값 개수 & 결측치 비율
cat_summary = pd.DataFrame({
    "n_unique": df[cat_cols].nunique(dropna=True),
    "n_missing": df[cat_cols].isna().sum(),
    "missing_pct": (df[cat_cols].isna().mean() * 100).round(2)
}).sort_values("n_unique")

print("\n범주형 변수 요약:")
print(cat_summary)

# 3) 각 범주형 변수별 예시 값 (앞 5개)
print("\ 예시 값 확인:")
for c in cat_cols:
    examples = df[c].dropna().unique()[:5]
    print(f"- {c} (unique {df[c].nunique(dropna=True)}): {examples}")


✅ 범주형 컬럼 개수: 13
✅ 범주형 컬럼 목록:
 ['직장산업대분류코드', '직업대분류코드', '분류코드_연령성별8코드', '분류코드_연령성별9코드', '분류코드_연령교육정도별1코드', '분류코드_장애인복지카드_소유여부', '분류코드_기부여부', '분류코드_현금기부여부', '분류코드_물품기부여부', '분류코드_자원봉사여부', '분류코드_계층의식코드', '분류코드_주관적소득수준코드', '분류코드_소득만족도코드']

📊 범주형 변수 요약:
                   n_unique  n_missing  missing_pct
분류코드_장애인복지카드_소유여부         2          0         0.00
분류코드_기부여부                 2          0         0.00
분류코드_현금기부여부               2          0         0.00
분류코드_물품기부여부               2          0         0.00
분류코드_자원봉사여부               2          0         0.00
분류코드_계층의식코드               3          0         0.00
분류코드_주관적소득수준코드            3      92385        49.96
분류코드_소득만족도코드              3      37463        20.26
직업대분류코드                  10      78626        42.52
분류코드_연령성별9코드             14      89197        48.24
분류코드_연령교육정도별1코드          15      14132         7.64
분류코드_연령성별8코드             16      60598        32.77
직장산업대분류코드                22      47908        25.91

📌 예시 값 확인:
- 직장산업대분류코드 

In [ ]:
# 직장산업대분류코드, 직업대분류코드 → 원핫인코딩

#만연령 제외, 이름에 "연령"이 들어간 모든 컬럼 삭제

In [ ]:
# ---------- 1) '연령' 컬럼 처리 ----------
# '만연령'은 유지하고, 나머지 '연령' 들어간 컬럼 삭제
drop_age_cols = [c for c in df.columns if ("연령" in c and c != "만연령")]
df = df.drop(columns=drop_age_cols)

print("🗑 삭제된 연령 관련 컬럼:", drop_age_cols)

# ---------- 2) 특정 코드 컬럼 원핫인코딩 ----------
onehot_cols = ["직장산업대분류코드", "직업대분류코드"]

# 실제 존재하는 컬럼만 대상으로 처리
onehot_cols = [c for c in onehot_cols if c in df.columns]

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)

print("One-Hot Encoding 완료:", onehot_cols)
print(" 최종 Shape:", df.shape)

🗑 삭제된 연령 관련 컬럼: ['분류코드_연령1코드', '분류코드_연령3코드', '분류코드_연령4코드', '분류코드_연령13코드', '분류코드_연령성별8코드', '분류코드_연령성별9코드', '분류코드_연령교육정도별1코드']
One-Hot Encoding 완료: ['직장산업대분류코드', '직업대분류코드']
 최종 Shape: (184915, 193)


In [ ]:
output_path = "df_after_age_onehot.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")